In [ ]:
import pandas as pd
import numpy as np

print("Processing Visa Data & Clustering GDP...")

# 1. Processing the Visa Data (Dynamic Header Finding)
visa_files = {
    2016: 'Visa-statistics-for-consulates-2016_en.xlsx',
    2017: 'Visa-statistics-for-consulates-2017_en.xlsx',
    2018: '2018-consulates-schengen-visa-stats.xlsx',
    2019: '2019-consulates-schengen-visa-stats.xlsx',
    2020: 'Visa-statistics-for-consulates-2020_en.xlsx',
    2021: '2021-consulates-schengen-visa-stats.xlsx',
    2022: 'Visa statistics for consulates in 2022_en.xlsx',
    2023: '2023-schengen-visa-statistics-consulates_en.xlsx.xlsx',
    2024: '2024-schengen-visa-statistics-consulates_en.xlsx'
}

processed_visa_data = []

for year, filename in visa_files.items():
    try:
        xls = pd.ExcelFile(filename)
        target_sheet = next((s for s in xls.sheet_names if 'Totals - Schengen' in s or 'Summary by Schengen' in s), xls.sheet_names[1])

        df_v = pd.read_excel(filename, sheet_name=target_sheet)
        df_v.columns = [str(c).lower().strip() for c in df_v.columns]

        if not any('schengen state' in c for c in df_v.columns):
            for idx, row in df_v.iterrows():
                if any('schengen state' in str(val).lower() for val in row.values):
                    df_v.columns = [str(val).lower().strip() for val in row.values]
                    df_v = df_v.iloc[idx+1:]
                    break

        state_col = [c for c in df_v.columns if 'schengen state' in c][0]
        app_col = [c for c in df_v.columns if 'applied for' in c and 'uniform' in c][0]
        issue_col = [c for c in df_v.columns if 'visas issued' in c and 'uniform' in c][0]

        df_subset = df_v[[state_col, app_col, issue_col]].copy()
        df_subset.columns = ['Country', 'Visa_Apps', 'Visa_Issued']
        df_subset = df_subset.dropna(subset=['Country'])
        df_subset = df_subset[~df_subset['Country'].astype(str).str.contains('Total', case=False)]

        df_subset['Visa_Apps'] = pd.to_numeric(df_subset['Visa_Apps'], errors='coerce')
        df_subset['Visa_Issued'] = pd.to_numeric(df_subset['Visa_Issued'], errors='coerce')
        df_subset['Visa_Acceptance_Rate'] = (df_subset['Visa_Issued'] / df_subset['Visa_Apps']) * 100
        df_subset['Year'] = year

        processed_visa_data.append(df_subset)
    except Exception as e:
        print(f"Skipping {year} due to format variation.")

visa_all = pd.concat(processed_visa_data, ignore_index=True)
name_map = {'Czech Republic': 'Czechia', 'Slovak Republic': 'Slovakia'}
visa_all['Country'] = visa_all['Country'].replace(name_map)

# 2. Merge with Master Data
master_df = pd.read_csv('master_dataset_EU_only.csv')
df_merged = pd.merge(master_df, visa_all[['Country', 'Year', 'Visa_Acceptance_Rate']], on=['Country', 'Year'], how='inner')

# 3. Create GDP Quartile Clusters
# Calculate the average GDP per country so their tier assignment remains stable across the 9 years
avg_gdp = df_merged.groupby('Country')['GDP_pc'].mean().reset_index()

# Use pd.qcut to divide countries into 4 equal groups based on GDP percentiles
cluster_labels = ['Lowest 25%', 'Higher Low 25-50%', 'Lower High 50-75%', 'Highest 25%']
avg_gdp['GDP_Cluster'] = pd.qcut(avg_gdp['GDP_pc'], q=4, labels=cluster_labels)

# Merge the cluster labels back into the main dataframe
df_final = pd.merge(df_merged, avg_gdp[['Country', 'GDP_Cluster']], on='Country', how='left')

# Save Enriched Data
df_final.to_csv('Enriched_Flight_Visa_Data.csv', index=False)
print(f"Success! Data saved with GDP Clusters. Total rows: {len(df_final)}")